### script de split des images pour dataset

In [ ]:
import os
import random
from PIL import Image
import pandas as pd

EXCEL_FILE = "/Users/damiencatheline/Documents/Lorcana/Lorcana_Database.xlsx"
DATA_DIR = "/Users/damiencatheline/Documents/Lorcana/Cartes_scan/Data/images"
df = pd.read_excel(EXCEL_FILE)

df = df.rename(columns={"Tri set": "File_name"})
df.drop(df[(df['Rareté'] != "C") & (df['Rareté'] != "U") & (df['Rareté'] != "R") & (df['Rareté'] != "SR") & (df['Rareté'] != "L")].index, inplace=True)
df.drop(df[(df['Set'] > 11) | (df['Encre #'].str.len() > 1) | (df['Numéro'] == 223) | (df['Numéro'] == 224) | (df['Numéro'] == 225)].index, inplace=True)

df["File_name"] = df["File_name"].apply(lambda x: f"{x}.jpg")
df['Encrable'] = df['Encrable'].apply(lambda x: 1 if x == "✔" else 0)
df["Catégorie"] = df["Catégorie"].apply(lambda x: "Personnage" if x == "①" else ("Action" if x == "②" else ("Action" if x == "③" else ("Objet" if x == "④" else "Lieu"))))
df["New_File_name"] = ''



for index, row in df.iterrows():
    new_file_name = f"{int(row['Set']):02d}_FR_{row['Numéro']}.jpg"
    df.at[index, "New_File_name"] = new_file_name
    row["New_File_name"] = new_file_name
    
    old_path = os.path.join(DATA_DIR, row["File_name"])
    new_path = os.path.join(DATA_DIR, row["New_File_name"])
    if os.path.exists(old_path):
        os.rename(old_path, new_path)
    else:
        print(f"⚠️ Fichier manquant: {old_path} à l'index {index}")

df = df.loc[:, ["New_File_name", "Set", "Numéro", "File_name", "Rareté", "Encre", "Encre #", "Nom FR", "Sous-titre FR", "Catégorie", "Classifications", "Encrable", "Coût", "Force", "Volonté", "Lore"]]

df.to_csv("Lorcana_Database.csv", index=False)

print(df.columns)

⚠️ Fichier manquant: /Users/damiencatheline/Documents/Lorcana/Cartes_scan/Data/images/01•FR•001 {U} - ❶① Coût 04✔ - ✟.jpg à l'index 0
⚠️ Fichier manquant: /Users/damiencatheline/Documents/Lorcana/Cartes_scan/Data/images/01•FR•002 {SR} - ❶① Coût 03✔ - ✟.jpg à l'index 1
⚠️ Fichier manquant: /Users/damiencatheline/Documents/Lorcana/Cartes_scan/Data/images/01•FR•003 {U} - ❶① Coût 04✔ - ☥.jpg à l'index 2
⚠️ Fichier manquant: /Users/damiencatheline/Documents/Lorcana/Cartes_scan/Data/images/01•FR•004 {U} - ❶① Coût 05✔ - ✟.jpg à l'index 3
⚠️ Fichier manquant: /Users/damiencatheline/Documents/Lorcana/Cartes_scan/Data/images/01•FR•005 {R} - ❶① Coût 08✖ - ✟.jpg à l'index 4
⚠️ Fichier manquant: /Users/damiencatheline/Documents/Lorcana/Cartes_scan/Data/images/01•FR•006 {R} - ❶① Coût 04✖ - ✟.jpg à l'index 5
⚠️ Fichier manquant: /Users/damiencatheline/Documents/Lorcana/Cartes_scan/Data/images/01•FR•007 {C} - ❶① Coût 01✔ - ✟.jpg à l'index 6
⚠️ Fichier manquant: /Users/damiencatheline/Documents/Lorcana

/Users/damiencatheline/Documents/Travail/formation Alyra/FormationIA/DL/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [ ]:
INPUT_DIR = "/Users/damiencatheline/Documents/Lorcana/Cartes_scan/Data/images"
OUTPUT_DIR = "/Users/damiencatheline/Documents/Lorcana/Cartes_scan/Data/images_resized"

os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE = (224, 224)
SPLIT_RATIO = (0.7, 0.15, 0.15)

random.seed(42)

def is_image(filename):
    return filename.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))


def resize_with_padding(img, target_size):
    img.thumbnail(target_size)  # conserve ratio

    new_img = Image.new("RGB", target_size, (0, 0, 0))  # fond noir

    # centrer l'image
    x_offset = (target_size[0] - img.size[0]) // 2
    y_offset = (target_size[1] - img.size[1]) // 2

    new_img.paste(img, (x_offset, y_offset))
    return new_img

# récupérer toutes les images
images = [f for f in os.listdir(INPUT_DIR) if is_image(f) and not f.startswith(".")]
random.shuffle(images)

# split
train_split = int(len(images) * SPLIT_RATIO[0])
val_split = int(len(images) * (SPLIT_RATIO[0] + SPLIT_RATIO[1]))

splits = {
    "train": images[:train_split],
    "val": images[train_split:val_split],
    "test": images[val_split:]
}

# traitement
for split_name, split_images in splits.items():
    split_path = os.path.join(OUTPUT_DIR, split_name)
    os.makedirs(split_path, exist_ok=True)
    

    for filename in split_images:
        input_path = os.path.join(INPUT_DIR, filename)
        output_path = os.path.join(split_path, filename)

        try:
            img = Image.open(input_path).convert("RGB")
            img_resized = resize_with_padding(img, IMG_SIZE)
            img_resized.save(output_path, quality=95)

        except Exception as e:
            print(f"❌ Erreur {filename}: {e}")

    df_split = df[df["New_File_name"].isin(split_images)]
    df_split.to_csv(f"{OUTPUT_DIR}/{split_name}_labels.csv", index=False)

    print(f"✅ {split_name} : {len(split_images)} images")

print("\n🎉 Resize + split terminé !")



✅ train : 1490 images
✅ val : 319 images
✅ test : 320 images

🎉 Resize + split terminé !
